# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

> You do not need to run the following cells if you are running this notebook locally. 

In [5]:
#!pip install -qU langchain langchain-openai langchain-cohere rank_bm25

We're also going to be leveraging [Qdrant's](https://qdrant.tech/documentation/frameworks/langchain/) (pronounced "Quadrant") VectorDB in "memory" mode (so we can leverage it locally in our colab environment).

In [6]:
#!pip install -qU qdrant-client

We'll also provide our OpenAI key, as well as our Cohere API key.

In [25]:
# api loading utilities

%run ../utils.py 


In [143]:
try:
    set_api_key_if_not_present("OPENAI_API_KEY")
    set_api_key_if_not_present("COHERE_API_KEY")
    set_api_key_if_not_present("RAGAS_APP_TOKEN")
    
except:
    import os
    import getpass
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")
    os.environ["OPENAI_API_KEY"] = getpass.getpass("openAI API Key:")
    
    

## Task 2: Data Collection and Preparation

We'll be using some reviews from the 4 movies in the John Wick franchise today to explore the different retrieval strategies.

These were obtained from IMDB, and are available in the [AIM Data Repository](https://github.com/AI-Maker-Space/DataRepository).

### Data Collection

We can simply `wget` these from GitHub.

You could use any review data you wanted in this step - just be careful to make sure your metadata is aligned with your choice.

In [9]:
!wget https://raw.githubusercontent.com/AI-Maker-Space/DataRepository/main/jw1.csv -O john_wick_1.csv
!wget https://raw.githubusercontent.com/AI-Maker-Space/DataRepository/main/jw2.csv -O john_wick_2.csv
!wget https://raw.githubusercontent.com/AI-Maker-Space/DataRepository/main/jw3.csv -O john_wick_3.csv
!wget https://raw.githubusercontent.com/AI-Maker-Space/DataRepository/main/jw4.csv -O john_wick_4.csv

--2025-05-20 15:11:45--  https://raw.githubusercontent.com/AI-Maker-Space/DataRepository/main/jw1.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 19628 (19K) [text/plain]
Saving to: ‘john_wick_1.csv’

john_wick_1.csv     100%[===================>]  19.17K  --.-KB/s    in 0.005s  

2025-05-20 15:11:45 (4.00 MB/s) - ‘john_wick_1.csv’ saved [19628/19628]

--2025-05-20 15:11:45--  https://raw.githubusercontent.com/AI-Maker-Space/DataRepository/main/jw2.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14747 (14K) [text/plain]
Sa

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

- Self-Query: Wants as much metadata as we can provide
- Time-weighted: Wants temporal data

> NOTE: While we're creating a temporal relationship based on when these movies came out for illustrative purposes, it needs to be clear that the "time-weighting" in the Time-weighted Retriever is based on when the document was *accessed* last - not when it was created.

In [10]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

documents = []

for i in range(1, 5):
  loader = CSVLoader(
      file_path=f"john_wick_{i}.csv",
      metadata_columns=["Review_Date", "Review_Title", "Review_Url", "Author", "Rating"]
  )

  movie_docs = loader.load()
  for doc in movie_docs:

    # Add the "Movie Title" (John Wick 1, 2, ...)
    doc.metadata["Movie_Title"] = f"John Wick {i}"

    # convert "Rating" to an `int`, if no rating is provided - assume 0 rating
    doc.metadata["Rating"] = int(doc.metadata["Rating"]) if doc.metadata["Rating"] else 0

    # newer movies have a more recent "last_accessed_at"
    doc.metadata["last_accessed_at"] = datetime.now() - timedelta(days=4-i)

  documents.extend(movie_docs)

Let's look at an example document to see if everything worked as expected!

In [11]:
documents[0]

Document(metadata={'source': 'john_wick_1.csv', 'row': 0, 'Review_Date': '6 May 2015', 'Review_Title': ' Kinetic, concise, and stylish; John Wick kicks ass.\n', 'Review_Url': '/review/rw3233896/?ref_=tt_urv', 'Author': 'lnvicta', 'Rating': 8, 'Movie_Title': 'John Wick 1', 'last_accessed_at': datetime.datetime(2025, 5, 17, 15, 11, 47, 135849)}, page_content=": 0\nReview: The best way I can describe John Wick is to picture Taken but instead of Liam Neeson it's Keanu Reeves and instead of his daughter it's his dog. That's essentially the plot of the movie. John Wick (Reeves) is out to seek revenge on the people who took something he loved from him. It's a beautifully simple premise for an action movie - when action movies get convoluted, they get bad i.e. A Good Day to Die Hard. John Wick gives the viewers what they want: Awesome action, stylish stunts, kinetic chaos, and a relatable hero to tie it all together. John Wick succeeds in its simplicity.")

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "JohnWick".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [12]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    documents,
    embeddings,
    location=":memory:",
    collection_name="JohnWick"
)

/home/mbudisic/Documents/AIE6/13_Advanced_Retrieval/.venv/lib/python3.13/site-packages/qdrant_client/http/models/models.py:758: SyntaxWarning: invalid escape sequence '\&'
  description="Check that the field is empty, alternative syntax for `is_empty: \&quot;field_name\&quot;`",
/home/mbudisic/Documents/AIE6/13_Advanced_Retrieval/.venv/lib/python3.13/site-packages/qdrant_client/http/models/models.py:762: SyntaxWarning: invalid escape sequence '\&'
  description="Check that the field is null, alternative syntax for `is_null: \&quot;field_name\&quot;`",


## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [18]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [19]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [20]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")


### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [21]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [26]:
naive_retrieval_chain.invoke({"question" : "Did people generally like John Wick?"})["response"].content | markdown_pipe

Based on the reviews provided, people generally liked John Wick. Many reviewers gave it high ratings, praised its stylish action, choreography, and overall entertainment value. Several reviews mention that it is a must-see for action fans and highlight its unique elements, style, and the performance of Keanu Reeves. However, there are some mixed opinions, with a few reviewers giving average or lower ratings and expressing less enthusiasm. Overall, the trend suggests that the majority of people who reviewed the film enjoyed it.

In [27]:
naive_retrieval_chain.invoke({"question" : "Do any reviews have a rating of 10? If so - can I have the URLs to those reviews?"})["response"].content | markdown_pipe

Yes, there are reviews with a rating of 10. The URL to the review with a rating of 10 is:

- [https://www.example.com/review/rw4854296/?ref_=tt_urv](https://www.example.com/review/rw4854296/?ref_=tt_urv)

(Note: The review URL corresponding to the rating of 10 in the data is "/review/rw4854296/?ref_=tt_urv".)

In [28]:
naive_retrieval_chain.invoke({"question" : "What happened in John Wick?"})["response"].content | markdown_pipe

In John Wick, a retired hitman named John Wick (played by Keanu Reeves) seeks revenge after his beloved dog is killed, his house is destroyed, and his car is stolen by a young Russian punk and his goons. The attack is carried out by the punk, who is connected to his father, a Russian mobster, making the conflict a personal and ruthless vendetta. As Wick re-enters the world of violence and assassination, he unleashes a relentless wave of destruction against those who wronged him, drawing the attention of the criminal underworld and bounty hunters. Throughout the series, Wick's story involves themes of revenge, consequences, and his struggle for peace amidst ongoing violence.

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [29]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(documents)

We'll construct the same chain - only changing the retriever.

In [30]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [31]:
bm25_retrieval_chain.invoke({"question" : "Did people generally like John Wick?"})["response"].content | markdown_pipe

Based on the reviews provided, people's opinions on John Wick are mixed. Some reviewers highly praise the first film, calling it stylish, fun, and a must-see for action fans. However, opinions on subsequent installments vary, with some viewers expressing disappointment or criticism, especially regarding the third film. Overall, it appears that many people did enjoy the first John Wick movie, but opinions on the later films are more varied.

In [32]:
bm25_retrieval_chain.invoke({"question" : "Do any reviews have a rating of 10? If so - can I have the URLs to those reviews?"})["response"].content | markdown_pipe

Based on the provided reviews, there are no reviews with a rating of 10.

In [33]:
bm25_retrieval_chain.invoke({"question" : "What happened in John Wick?"})["response"].content | markdown_pipe

In the John Wick film series, the story revolves around John Wick, a retired hitman who is drawn back into the criminal underworld. The first movie, "John Wick," showcases how Wick seeks vengeance after thugs steal his car and kill his dog, which was a final gift from his deceased wife. The series features intense, highly choreographed action scenes, with Wick taking on numerous enemies using hand-to-hand combat and firearms. The subsequent films, "John Wick 2," "John Wick 3," and "John Wick 4," deepen the story of his ongoing battles with assassins, criminal organizations, and those who seek to control or eliminate him. Throughout the series, themes of revenge, loyalty, and survival are prominent, along with stylized violence and elaborate world-building involving a secret society of assassins.

It's not clear that this is better or worse - but the `I don't know` isn't great!

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [44]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-english-v3.0")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [45]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [46]:
contextual_compression_retrieval_chain.invoke({"question" : "Did people generally like John Wick?"})["response"].content | markdown_pipe

Based on the reviews provided, people generally liked John Wick. The positive reviews describe it as an "insanely fun," "slick," "brilliantly shot," and "highly recommended" action film, with high ratings such as 9 and 10 out of 10. However, there is a less favorable review for the third film, giving it a rating of 5, indicating some disappointment. Overall, the majority of the reviews suggest that people generally liked the movie.

In [47]:
contextual_compression_retrieval_chain.invoke({"question" : "Do any reviews have a rating of 10? If so - can I have the URLs to those reviews?"})["response"].content | markdown_pipe

Yes, there are reviews with a rating of 10. Here are the URLs to those reviews:

1. [Review on john_wick_3.csv](https://yourdomain.com/review/rw4854296/?ref_=tt_urv) titled "A Masterpiece & Brilliant Sequel"

2. [Review on john_wick_3.csv](https://yourdomain.com/review/rw4860412/?ref_=tt_urv) titled "It's got its own action style!"

Please note that these URLs are based on the review data provided; replace "yourdomain.com" with the actual website domain if necessary.

In [48]:
contextual_compression_retrieval_chain.invoke({"question" : "What happened in John Wick?"})["response"].content | markdown_pipe

In the John Wick movies, John Wick, a retired hitman, seeks revenge after personal tragedies. In the first film, his beloved dog is killed and his car is stolen, which motivates him to come out of retirement to hunt down those responsible. He unleashes a brutal and relentless campaign against gangsters and criminals who cross his path, driven by vengeance. The second film continues with Wick being drawn back into the criminal underworld when Santino D'Antonio, a mobster, asks for his help to eliminate his sister and gain power, which leads to further violence and a bounty on Wick's head. Overall, the series is characterized by intense action, stylish combat sequences, and Wick's unwavering resolve to settle personal scores.

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [49]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [50]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [51]:
multi_query_retrieval_chain.invoke({"question" : "Did people generally like John Wick?"})["response"].content | markdown_pipe

Based on the reviews provided, people generally liked John Wick. Many reviews rate the movies highly, praising their action sequences, style, and entertainment value. For example, some reviews gave ratings of 9 or 10 out of 10 and described the films as "slick," "insanely fun," and "remarkable." Overall, the sentiment indicates that John Wick is well-liked by audiences who appreciate its action and style.

In [52]:
multi_query_retrieval_chain.invoke({"question" : "Do any reviews have a rating of 10? If so - can I have the URLs to those reviews?"})["response"].content | markdown_pipe

Yes, there are reviews with a rating of 10. The URLs to those reviews are:

1. [https://yourwebsite.com/review/rw4854296/?ref_=tt_urv](https://yourwebsite.com/review/rw4854296/?ref_=tt_urv) (Review for John Wick 3 titled "A Masterpiece & Brilliant Sequel")
2. [https://yourwebsite.com/review/rw5503708/?ref_=tt_urv](https://yourwebsite.com/review/rw5503708/?ref_=tt_urv) (Review for John Wick 1 titled "love this movie highly recommend")

In [53]:
multi_query_retrieval_chain.invoke({"question" : "What happened in John Wick?"})["response"].content | markdown_pipe

In the John Wick film series, the story revolves around John Wick, a retired hitman who is drawn back into the violent underworld after personal tragedies. The first movie depicts how Wick, mourning the loss of his wife and acting out of revenge for the killing of his dog and the theft of his car, unleashes a relentless and highly skilled rampage against gangsters and assassins. Throughout the series, he faces numerous enemies, navigates complex criminal rules, and deals with the consequences of his actions, all while showcasing spectacular action sequences. Each installment explores his ongoing struggles and the dangerous criminal world he once belonged to.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [63]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = documents
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [64]:
client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

from langchain_qdrant import Qdrant

parent_document_vectorstore = Qdrant(
    collection_name="full_documents", embeddings=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [65]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [66]:
if len(parent_document_retriever.docstore.docs) == 0: # don't attempt to store them twice
    parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [67]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [68]:
parent_document_retrieval_chain.invoke({"question" : "Did people generally like John Wick?"})["response"].content | markdown_pipe

Based on the provided reviews, people's opinions on John Wick vary. Some reviews are very positive, highlighting the series as highly recommended and well-made, with praise for the action and emotional setup. For example, one reviewer said they "love this movie" and "highly recommend" it, and another stated the series remained "remarkably consistent and well received." 

However, there is at least one negative review that criticizes John Wick 4 harshly, calling it "HORRIBLE" and criticizing its plot, fight scenes, and realism. 

Overall, while many seem to like the series, opinions on the latest installment, John Wick 4, appear mixed, with some fans liking it and others disliking it.

In [69]:
parent_document_retrieval_chain.invoke({"question" : "Do any reviews have a rating of 10? If so - can I have the URLs to those reviews?"})["response"].content | markdown_pipe

Yes, there is a review with a rating of 10. 

The URL to that review is: /review/rw4854296/?ref_=tt_urv

In [70]:
parent_document_retrieval_chain.invoke({"question" : "What happened in John Wick?"})["response"].content | markdown_pipe

In the John Wick movies, John Wick is a retired assassin who is drawn back into a violent world of killing and revenge. In the first film, he comes out of retirement after a gang kills his dog and steals his car, prompting a relentless, bloody vendetta against those who wronged him. The second film continues his story as he is forced to help another criminal boss take over an assassin's guild, leading to more action, many kills, and international adventures. Overall, the series depicts John Wick's deadly skills, his desire for peace, and the consequences of a violent past catching up with him.

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [71]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [72]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [73]:
ensemble_retrieval_chain.invoke({"question" : "Did people generally like John Wick?"})["response"].content | markdown_pipe

Based on the reviews provided, people generally liked John Wick. The reviews are mostly positive, highlighting its stylish action, exciting sequences, and entertainment value. For example, reviews mention that it is "a must-see for action fans," "slick, violent fun," and "an insanely fun" film. Several reviewers rated it highly (e.g., 8/10, 9/10, 10/10) and recommend it to action enthusiasts. While there are some mixed or negative opinions, the overall sentiment from the collected reviews suggests that people generally enjoyed John Wick.

In [74]:
ensemble_retrieval_chain.invoke({"question" : "Do any reviews have a rating of 10? If so - can I have the URLs to those reviews?"})["response"].content | markdown_pipe

Yes, there are reviews with a rating of 10. The URL to a review with a rating of 10 is: /review/rw4854296/?ref_=tt_urv

In [75]:
ensemble_retrieval_chain.invoke({"question" : "What happened in John Wick?"})["response"].content | markdown_pipe

In the John Wick film series, the story revolves around John Wick, a retired assassin who is pulled back into a violent underworld of crime and assassins. The first film depicts how Wick comes out of retirement after gangsters kill his dog and take his car, seeking revenge and unleashing a deadly rampage against those who wronged him. Subsequent films explore his ongoing conflicts with various criminal organizations, his efforts to adhere to and challenge the rules of the assassin world, and the consequences of his actions, often featuring elaborate and choreographed action sequences. Overall, the series combines themes of revenge, loyalty, and the high-stakes criminal underworld, with John Wick navigating through battles against numerous enemies to find peace or exert retribution.

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

> NOTE: You do not need to run this cell if you're running this locally

In [77]:
#!pip install -qU langchain_experimental

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [78]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [79]:
semantic_documents = semantic_chunker.split_documents(documents)

Let's create a new vector store.

In [80]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="JohnWickSemantic"
)

We'll use naive retrieval for this example.

In [81]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [82]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results! 🏗️🚢🚀

In [83]:
semantic_retrieval_chain.invoke({"question" : "Did people generally like John Wick?"})["response"].content | markdown_pipe

Based on the reviews provided, people generally liked John Wick. The reviews are mostly positive, with ratings frequently in the 8 or 9 out of 10 range, and the comments describe the movies as stylish, fun, and well-choreographed action films. While there are some less favorable opinions, the overall sentiment indicates that many viewers appreciated and enjoyed the series.

In [84]:
semantic_retrieval_chain.invoke({"question" : "Do any reviews have a rating of 10? If so - can I have the URLs to those reviews?"})["response"].content | markdown_pipe

Yes, there is a review with a rating of 10. The URL to that review is /review/rw4854296/?ref_=tt_urv.

In [85]:
semantic_retrieval_chain.invoke({"question" : "What happened in John Wick?"})["response"].content | markdown_pipe

In the movie John Wick, the story centers around a retired assassin named John Wick, played by Keanu Reeves. The film begins with Wick living peacefully after leaving his violent past behind. However, his quiet life is disrupted when a young Russian-American punk, who notices Wick's vintage car, tries to buy it. Wick declines, but shortly afterward, the punk and his goons surprise him at his home, beat him up, kill his dog, and steal his car. It is revealed that Wick is a legendary hitman, and the attack on his dog, a gift from his late wife, deeply enrages him.

This brutal assault ignites Wick's quest for revenge. As the story unfolds, Wick re-enters the violent underworld he had left, seeking justice against those who wronged him. The film features intense action sequences, stylish stunts, and explores Wick's character as he wages a relentless war against the gangsters and mobsters involved in the attack, especially focusing on the Russian mobster whose son was responsible. The movie highlights themes of vengeance, loyalty, and the consequences of a violent past catching up with someone who tries to find peace.

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [174]:
from uuid import uuid4
set_api_key_if_not_present("LANGCHAIN_API_KEY", "LANGCHAIN_API_KEY:")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"AIM - AIE6.13 Retrievals - {uuid4().hex[0:8]}"

## Generate the Golden Dataset using Ragas

In [ ]:
from r

In [146]:
from ragas.testset import TestsetGenerator, Testset
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings


In [ ]:

# 2. Instantiate your generator
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)

dataset_count = 10
dataset_name = "john_wick_golden"
filename = f"./{dataset_name}_{dataset_count}.jsonl"

try:
    ragas_dataset = Testset.from_jsonl(filename)
except:
    # 3. Generate testset (this builds the KG internally)
    ragas_dataset = generator.generate_with_langchain_docs(documents, testset_size=dataset_count)
    ragas_dataset.to_jsonl(filename)
ragas_dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Could you provide a detailed description of th...,[: 0\nReview: The best way I can describe John...,John Wick can be described as a film similar t...,single_hop_specifc_query_synthesizer
1,Why John Wick movie so popular with all them a...,[: 2\nReview: With the fourth installment scor...,The John Wick series has become immensely popu...,single_hop_specifc_query_synthesizer
2,Can you explane why John Wick is so specail de...,[: 3\nReview: John wick has a very simple reve...,"John Wick stands out because, although it has ...",single_hop_specifc_query_synthesizer
3,Why Reeves look so good in John Wick even tho ...,[: 4\nReview: Though he no longer has a taste ...,"Reeves plays John Wick, a retired assassin who...",single_hop_specifc_query_synthesizer
4,"How does the character of John Wick, portrayed...",[<1-hop>\n\n: 10\nReview: Wow what a great sur...,"In the first film, John Wick, played by Keanu ...",multi_hop_abstract_query_synthesizer
5,"How does John Wick's quest for revenge, sparke...",[<1-hop>\n\n: 0\nReview: The best way I can de...,John Wick's quest for revenge begins after a y...,multi_hop_abstract_query_synthesizer
6,How do the choreographed violence and hardcore...,"[<1-hop>\n\n: 0\nReview: No doubt about it, ""J...","""John Wick: Chapter 2"" features choreographed ...",multi_hop_abstract_query_synthesizer
7,How do the themes of relentless violence and e...,[<1-hop>\n\n: 15\nReview: ...totally over-rate...,The theme of relentless violence is highlighte...,multi_hop_abstract_query_synthesizer
8,How do the creators of John Wick 3 combine cle...,[<1-hop>\n\n: 22\nReview: Lets contemplate abo...,The creators of John Wick 3 have crafted a uni...,multi_hop_specific_query_synthesizer
9,Why John Wick 2 no surprise like first one but...,[<1-hop>\n\n: 10\nReview: The first John Wick ...,John Wick 2 doesn't have the ability to sneak ...,multi_hop_specific_query_synthesizer


In [180]:
ragas_dataset.upload()

[2025-05-20 17:15:01 - (2025-05-20 21:15:01 UTC)] [ERROR] [ragas.utils] [RagasID: a-27f5a1a8245c4ec4952e7bc6811cc3f5, App-Version: 0.2.15] [API_ERROR] Request failed. Status Code: 409, URL: https://api.ragas.io/api/v1/alignment/testset, Error Message: 
API Message: Testset with run ID '46034449-9568-4c20-8987-47e6e5f205f6' already exists


UploadException: Request failed: 
API Message: Testset with run ID '46034449-9568-4c20-8987-47e6e5f205f6' already exists

## Run the chains on the responses

### Naive Retrieval Chain

```python
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)
```

---

### BM25 Retrieval Chain

```python
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(documents)

bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)
```

---

### Contextual Compression Retrieval Chain

```python
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-english-v3.0")
# naive_retriever is defined as vectorstore.as_retriever(search_kwargs={"k" : 10})
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)
```

---

### Multi-Query Retrieval Chain

```python
from langchain.retrievers.multi_query import MultiQueryRetriever

# naive_retriever is defined as vectorstore.as_retriever(search_kwargs={"k" : 10})
# chat_model is ChatOpenAI(model="gpt-4.1-nano")
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)
```

---

### Parent Document Retrieval Chain

```python
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models
from langchain_qdrant import Qdrant # Added this import based on usage

parent_docs = documents
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

# embeddings is OpenAIEmbeddings(model="text-embedding-3-small")
parent_document_vectorstore = Qdrant(
    collection_name="full_documents", embeddings=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

if len(parent_document_retriever.docstore.docs) == 0: # don't attempt to store them twice
    parent_document_retriever.add_documents(parent_docs, ids=None)

parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)
```

---

### Ensemble Retrieval Chain

```python
from langchain.retrievers import EnsembleRetriever

# retriever_list contains previously defined retrievers:
# [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)
```

---

### Semantic Retrieval Chain

```python
from langchain_experimental.text_splitter import SemanticChunker
# from langchain_community.vectorstores import Qdrant # Already imported
# from langchain_openai import OpenAIEmbeddings # Already imported

# embeddings is OpenAIEmbeddings(model="text-embedding-3-small")
semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

semantic_documents = semantic_chunker.split_documents(documents)

semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="JohnWickSemantic"
)

semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)
```


In [170]:
from typing import Any, Iterator, List
from langchain_core.runnables import RunnableSerializable, RunnableParallel
from ragas import EvaluationDataset
import asyncio
import nest_asyncio
nest_asyncio.apply()


def enter_chain( chain : RunnableSerializable, question:str) -> str:
    return chain.invoke({"question" : question})["response"].content 

def batch(iterable: List[Any], size: int = 16) -> Iterator[List[Any]]:
    for i in range(0, len(iterable), size):
        yield iterable[i : i + size]


async def aapply_rag_chain(
    rag_chain: RunnableSerializable,
    ragas_testset: Testset,
    batch_size: int = 16,
) -> EvaluationDataset:
    """
    Apply RAG chain to dataset items in parallel batches.

    Args:
        rag_chain: The RAG chain to apply
        ragas_ds: The dataset to process
        batch_size: Number of items to process in each batch
    """
    
    ragas_ds = ragas_testset.to_evaluation_dataset()

    async def process_item(item):
        response = await rag_chain.ainvoke({"question": item.user_input})
        item.response = response["response"].content
        item.retrieved_contexts = [
            context.page_content
            for context in response["context"]
        ]

    # Process items in batches using the batch function
    for batch_items in batch(list(ragas_ds), size=batch_size):
        tasks = [process_item(item) for item in batch_items]
        await asyncio.gather(*tasks)
        
    return ragas_ds

In [171]:
result  = await aapply_rag_chain(naive_retrieval_chain, ragas_dataset)

In [172]:
result.to_pandas()

,user_input,retrieved_contexts,reference_contexts,response,reference
0,Could you provide a detailed description of th...,"[: 9\nReview: At first glance, John Wick sound...",[: 0\nReview: The best way I can describe John...,Certainly! The movie *John Wick* (2014) is a s...,John Wick can be described as a film similar t...
1,Why John Wick movie so popular with all them a...,[: 20\nReview: John Wick is something special....,[: 2\nReview: With the fourth installment scor...,John Wick movies are so popular because they f...,The John Wick series has become immensely popu...
2,Can you explane why John Wick is so specail de...,[: 3\nReview: John wick has a very simple reve...,[: 3\nReview: John wick has a very simple reve...,John Wick is considered special despite its si...,"John Wick stands out because, although it has ..."
3,Why Reeves look so good in John Wick even tho ...,[: 20\nReview: John Wick is something special....,[: 4\nReview: Though he no longer has a taste ...,Reeves looks so good in John Wick even though ...,"Reeves plays John Wick, a retired assassin who..."
4,"How does the character of John Wick, portrayed...",[: 8\nReview: In this 2nd installment of John ...,[<1-hop>\n\n: 10\nReview: Wow what a great sur...,"The character of John Wick, portrayed by Keanu...","In the first film, John Wick, played by Keanu ..."
5,"How does John Wick's quest for revenge, sparke...",[: 0\nReview: The best way I can describe John...,[<1-hop>\n\n: 0\nReview: The best way I can de...,"John Wick's quest for revenge, initiated by th...",John Wick's quest for revenge begins after a y...
6,How do the choreographed violence and hardcore...,[: 16\nReview: John Wick Chapter 2 pits Keanu ...,"[<1-hop>\n\n: 0\nReview: No doubt about it, ""J...",The choreographed violence and hardcore action...,"""John Wick: Chapter 2"" features choreographed ..."
7,How do the themes of relentless violence and e...,"[: 9\nReview: At first glance, John Wick sound...",[<1-hop>\n\n: 15\nReview: ...totally over-rate...,"According to the contrasting reviews, the them...",The theme of relentless violence is highlighte...
8,How do the creators of John Wick 3 combine cle...,[: 3\nReview: John wick has a very simple reve...,[<1-hop>\n\n: 22\nReview: Lets contemplate abo...,The creators of John Wick 3 combine clear chor...,The creators of John Wick 3 have crafted a uni...
9,Why John Wick 2 no surprise like first one but...,[: 18\nReview: The first John Wick movie had a...,[<1-hop>\n\n: 10\nReview: The first John Wick ...,"Based on the reviews and context provided, the...",John Wick 2 doesn't have the ability to sneak ...


In [178]:
from typing import Dict


chains:Dict[str, RunnableSerializable] = {
    "naive":naive_retrieval_chain,
    "bm25":bm25_retrieval_chain,
    "compression":contextual_compression_retrieval_chain,
    "multi":multi_query_retrieval_chain,
    "parent":parent_document_retrieval_chain,
    "ensemble":ensemble_retrieval_chain,
    "semantic":semantic_retrieval_chain
}

responses = {}

for l, c in chains.items():
    print(l)    
    try:
        responses[l] = EvaluationDataset.from_jsonl(f"evaluation_datasets/{l}.jsonl")
    except:
        responses[l] = await aapply_rag_chain(c, ragas_dataset)
        responses[l].to_jsonl(f"evaluation_datasets/{l}.jsonl")



naive
bm25
compression
multi
parent
ensemble
semantic


In [186]:
from ragas.metrics import (
    LLMContextRecall,
    ContextEntityRecall,
    LLMContextPrecisionWithReference,
    NoiseSensitivity,
    ResponseRelevancy,
    FactualCorrectness,
    Faithfulness,
)
from ragas import evaluate
from ragas.evaluation import EvaluationResult

# instantiate with any custom modes you need
metrics = [
    # Pure retrieval quality
    LLMContextRecall(),                                # “context_recall”:contentReference[oaicite:0]{index=0}
    ContextEntityRecall(),                             # “context_entity_recall”:contentReference[oaicite:1]{index=1}
    LLMContextPrecisionWithReference(),                # “context_precision”:contentReference[oaicite:2]{index=2}

    # Robustness / stress-testing retrieval
    NoiseSensitivity(mode="relevant"),                 # “noise_sensitivity” (relevant):contentReference[oaicite:3]{index=3}

    # End-to-end answer quality
    ResponseRelevancy(),                               # “answer_relevancy”:contentReference[oaicite:4]{index=4}
    FactualCorrectness(mode="f1"),                     # “factual_correctness”:contentReference[oaicite:5]{index=5}
    Faithfulness(),                                    # “faithfulness”:contentReference[oaicite:6]{index=6}
]


In [ ]:
EvaluationResult

In [ ]:
from datasets import load_from_disk

results = {}
scores= {}
for l, c in responses.items():
    print(l)    
    try:
        scores[l] = load_from_disk(f"results/{l}.jsonl")
    except:
        results[l] = evaluate(responses[l], metrics=metrics, experiment_name=f"JohnWick_{l}")
        scores[l] = results[l].scores
        results[l].scores.save_to_disk(f"results/{l}.jsonl")

